# Reference · Day 2 studio — the smallest honest pipeline

**Not a marking key.** There is no single right answer to "build the smallest honest
pipeline you can defend" — you chose your own columns, and so did everyone else.

This is here so you can compare. Run it, read it, and check your own notebook against
the list at the bottom. If yours differs, the interesting question is *why*, and that
is a better conversation than "was I right".

In [ ]:
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from stat764 import load

ames = load("ames.csv")

## One reference implementation

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric     = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF"]
categorical = ["Neighborhood", "Central_Air", "Garage_Type"]

X_train, X_test, y_train, y_test = train_test_split(
    ames[numeric + categorical], ames["SalePrice"], test_size=0.25, random_state=764)

pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("fill",   SimpleImputer(strategy="median")),
                          ("scale",  StandardScaler())]),                      numeric),
        ("cat", Pipeline([("fill",   SimpleImputer(strategy="constant",
                                                   fill_value="Missing")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])),
    ("model", LinearRegression()),
])

pipe.fit(X_train, y_train)

baseline = np.full(len(y_test), y_train.mean())
print(f"  mean baseline        R-squared {r2_score(y_test, baseline):.3f}")
print(f"  this pipeline        R-squared {r2_score(y_test, pipe.predict(X_test)):.3f}")
print(f"  columns in {len(numeric) + len(categorical)}, "
      f"after encoding {pipe.named_steps['prep'].transform(X_train).shape[1]}")

## Your answer probably differs, and that is fine

Here is the range the same skeleton produces with different column choices. Find where
yours sits.

In [ ]:
allnum = [c for c in ames.select_dtypes("number").columns if c != "SalePrice"]
allcat = list(ames.select_dtypes(include="object").columns)

def try_columns(num, cat):
    X = ames[num + cat]
    Xtr, Xte, ytr, yte = train_test_split(
        X, ames["SalePrice"], test_size=0.25, random_state=764)
    steps = []
    if num:
        steps.append(("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                                       ("scale", StandardScaler())]), num))
    if cat:
        steps.append(("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                              fill_value="Missing")),
                                       ("encode", OneHotEncoder(handle_unknown="ignore"))]), cat))
    p = Pipeline([("prep", ColumnTransformer(steps)),
                  ("model", LinearRegression())]).fit(Xtr, ytr)
    return r2_score(yte, p.predict(Xte)), p.named_steps["prep"].transform(Xtr[:5]).shape[1]

print(f"  {'columns chosen':<34}{'in':>4}{'encoded':>9}{'R2':>8}")
for label, num, cat in [
    ("one predictor",            ["Gr_Liv_Area"], []),
    ("three numeric",            ["Gr_Liv_Area", "Overall_Qual", "Year_Built"], []),
    ("the reference above",      numeric, categorical),
    ("every numeric column",     allnum, []),
    ("everything in the file",   allnum, allcat),
]:
    r, n = try_columns(num, cat)
    print(f"  {label:<34}{len(num)+len(cat):>4}{n:>9}{r:>8.3f}")

Notice the shape of that: one good predictor gets you two thirds of the way, and the
next thirty-odd columns buy a fraction of what the first three did. Notice too that
*every numeric column* does worse than a smaller set that includes some categoricals.

More is not reliably better — which is the thread Meeting 3 picks up.

## Check your own notebook against this

Not "is my number as high as theirs". These:

| | |
|---|---|
| **1** | Did you call `train_test_split` **before** anything else touched the data? Scroll up and check — this is the one people get wrong while believing they did not. |
| **2** | Is every preprocessing step **inside** the `Pipeline`? If you have a `.fit_transform(` anywhere outside it, that is a leak. |
| **3** | Did you compute a **baseline** and beat it? If you never computed one, you do not actually know whether your model is doing anything. |
| **4** | Would you **put your number on the board** and defend where it came from? |
| **5** | Does the notebook run top to bottom after **Restart & Run All**? |

If any of 1, 2 or 5 is a no, fix it before Lab 2 — those three are the habits the rest
of the course is built on. If 3 or 4 is a no, that is worth two minutes of conversation
rather than a fix.

⚠ **A smaller model that passes all five is a better answer than a larger one that
fails any of them.** "Smallest you can defend" was the brief, and defensible beats big.